# 5. Machine Learning Models
Training and evaluation of Random Forest, XGBoost, and SVM classifiers with 10-fold cross-validation and 70/30 holdout test set.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, accuracy_score, cohen_kappa_score,
    f1_score, confusion_matrix, roc_curve, auc
)
import xgboost as xgb
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

## 1. Load Dataset / 

In [ ]:
# Pathlib Path for output directory
PROJECT_ROOT = Path.cwd().parent

df = pd.read_csv(PROJECT_ROOT / 'data' / 'samples' / 'dataset_training.csv')
FACTORS = ['elevation', 'slope', 'distance_to_river', 'distance_to_coast',
    'land_cover', 'soil_type', 'ndvi', 'rainfall']

X = df[FACTORS].values
y = df['label'].values

print(f'Dataset: {len(df)} samples ({sum(y==1)} flood, {sum(y==0)} non-flood)')
print(f'Features: {FACTORS}')

## 2. Train-Test Split & Cross-Validation / 

70/30 stratified split, then 10-fold stratified CV on training set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
print(f'Training set: {len(X_train)} samples')
print(f'Testing set: {len(X_test)} samples')

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=6, random_state=42
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='auc',
        use_label_encoder=False
    ),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
        probability=True, random_state=42))
    ]),
}

## 3. Model Training & Evaluation / 

In [ ]:
cv_results = {name: {'auc': [], 'acc': [], 'kappa': [], 'f1': [], 'tss': []}
    for name in models}
test_results = {}

for name, model in models.items():
    # 10-fold CV on training set
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    m = clone(model)
    m.fit(X_tr, y_tr)
    y_pred_val = m.predict(X_val)
    y_prob_val = m.predict_proba(X_val)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_val).ravel()

    cv_results[name]['auc'].append(roc_auc_score(y_val, y_prob_val))
    cv_results[name]['acc'].append(accuracy_score(y_val, y_pred_val))
    cv_results[name]['kappa'].append(cohen_kappa_score(y_val, y_pred_val))
    cv_results[name]['f1'].append(f1_score(y_val, y_pred_val))
    cv_results[name]['tss'].append((tp/(tp+fn)) + (tn/(tn+fp)) - 1)

    # Final evaluation on test set
    m_final = clone(model)
    m_final.fit(X_train, y_train)
    y_pred_test = m_final.predict(X_test)
    y_prob_test = m_final.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()

    test_results[name] = {
        'y_test': y_test.copy(),
        'y_pred': y_pred_test.copy(),
        'y_prob': y_prob_test.copy(),
        'auc': roc_auc_score(y_test, y_prob_test),
        'acc': accuracy_score(y_test, y_pred_test),
        'kappa': cohen_kappa_score(y_test, y_pred_test),
        'f1': f1_score(y_test, y_pred_test),
        'tss': (tp/(tp+fn)) + (tn/(tn+fp)) - 1,
        'cm': confusion_matrix(y_test, y_pred_test),
    }
    print(f'{name} complete')

# Keep trained RF for feature importance
rf_trained = clone(models['Random Forest'])
rf_trained.fit(X_train, y_train)

## 4. Cross-Validation Results / 

In [ ]:
cv_table = {}
for metric, label in [('auc', 'AUC-ROC'), ('acc', 'Accuracy'),
    ('kappa', 'Kappa'), ('f1', 'F1-Score'), ('tss', 'TSS')]:
        row = {}
        for name in models:
            vals = cv_results[name][metric]
            row[name] = f'{np.mean(vals):.3f} +/- {np.std(vals):.3f}'
            cv_table[label] = row

pd.DataFrame(cv_table).T

## 5. Test Set Results / 

In [ ]:
test_table = {}
for metric, label in [('auc', 'AUC-ROC'), ('acc', 'Accuracy'),
    ('kappa', 'Kappa'), ('f1', 'F1-Score'), ('tss', 'TSS')]:
        row = {}
        for name in models:
            row[name] = round(test_results[name][metric], 3)
            test_table[label] = row

test_df = pd.DataFrame(test_table).T
print(test_df.to_string())

# Save metrics
test_df.to_csv(PROJECT_ROOT / 'outputs' / 'model_metrics.csv')

## 6. Confusion Matrices / 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, name in enumerate(['Random Forest', 'XGBoost', 'SVM']):
    cm = test_results[name]['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Non-Flood', 'Flood'],
    yticklabels=['Non-Flood', 'Flood'],
    ax=axes[idx])
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_title(name)

plt.suptitle('Confusion Matrices (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs' / 'confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. ROC Curves / ROC

In [ ]:
plt.figure(figsize=(8, 7))
colors = {'Random Forest': 'green', 'XGBoost': 'blue', 'SVM': 'red'}

for name in models:
    y_true = test_results[name]['y_test']
    y_prob = test_results[name]['y_prob']
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[name], linewidth=2,
    label=f'{name} (AUC = {auc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve (Test Set)', fontsize=13)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs' / 'roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Feature Importance (Random Forest) / 

In [ ]:
importances = rf_trained.feature_importances_
indices = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(indices)), importances[indices], color='steelblue')
ax.set_yticks(range(len(indices)))
ax.set_yticklabels([FACTORS[i] for i in indices])
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Feature Importance - Random Forest')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs' / 'feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# Save feature importance
fi_df = pd.DataFrame({
    'Feature': FACTORS,
    'Importance': [round(v, 4) for v in importances]
}).sort_values('Importance', ascending=False)
fi_df.to_csv(PROJECT_ROOT / 'outputs' / 'feature_importance.csv', index=False)
print(fi_df.to_string(index=False))

## 9. Train Final RF on Full Dataset / RF

In [ ]:
rf_final = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_final.fit(X, y)

os.makedirs(PROJECT_ROOT / 'model_export', exist_ok=True)
joblib.dump(rf_final, PROJECT_ROOT / 'model_export' / 'rf_model.joblib')
print('Saved: model_export/rf_model.joblib')